In [ ]:
import os
import io
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image, ImageFilter, ImageEnhance
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
DATA_DIR = "./data/tiny-imagenet-200"
CHECKPOINT_DIR = "./checkpoints"
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

NUM_CLASSES = 200
BATCH_SIZE = 256
NUM_WORKERS = 4
STRATEGIES = ["baseline", "randaugment", "augmix"]

GPU_ID = 6

if torch.cuda.is_available():
    DEVICE = torch.device(f"cuda:{GPU_ID}")
    torch.cuda.set_device(DEVICE)
    print(f"Device: {DEVICE}  ({torch.cuda.get_device_name(DEVICE)})")

    # A100 only
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
else:
    DEVICE = torch.device("cpu")
    print("Device: cpu (no CUDA available)")

Device: cuda:6  (NVIDIA A100-SXM4-80GB)


In [ ]:
def create_model(num_classes=NUM_CLASSES):
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, num_classes)
    return model

# Load best checkpoints
trained_models = {}
for name in STRATEGIES:
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"best_{name}.pth")
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    model = create_model().to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    trained_models[name] = model
    print(f"{name}  (val acc = {ckpt['val_acc']:.2f}%, epoch {ckpt['epoch']})")

In [ ]:
NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
DENORM_MEAN = torch.tensor([0.485, 0.456, 0.406])
DENORM_STD  = torch.tensor([0.229, 0.224, 0.225])


class TinyImageNet(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples = []
        classes = sorted(
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d)) and d.startswith("n")
        )
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for cls in classes:
            img_dir = os.path.join(root_dir, cls, "images")
            if not os.path.isdir(img_dir):
                img_dir = os.path.join(root_dir, cls)
            for fname in sorted(os.listdir(img_dir)):
                if fname.lower().endswith((".jpeg", ".jpg", ".png")):
                    self.samples.append(
                        (os.path.join(img_dir, fname), self.class_to_idx[cls])
                    )
        print(f"{len(self.samples):,} images · {len(classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


class CorruptedDataset(Dataset):
    def __init__(self, base_dataset, corrupt_fn, tensor_transform):
        self.base = base_dataset
        self.corrupt_fn = corrupt_fn
        self.tensor_transform = tensor_transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        path, label = self.base.samples[idx]
        img = Image.open(path).convert("RGB")
        img = self.corrupt_fn(img)
        img = self.tensor_transform(img)
        return img, label


VAL_DIR = os.path.join(DATA_DIR, "val")
val_transform = transforms.Compose([transforms.ToTensor(), NORMALIZE])
val_dataset = TinyImageNet(VAL_DIR, transform=val_transform)
val_dataset_raw = TinyImageNet(VAL_DIR, transform=None)

tensor_transform = transforms.Compose([transforms.ToTensor(), NORMALIZE])

In [ ]:
def _clamp_pil(arr):
    return Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))


# Noise corruptions
def gaussian_noise(img, severity):
    sigma = [0.04, 0.08, 0.14, 0.20, 0.28][severity - 1]
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr += np.random.normal(0, sigma, arr.shape).astype(np.float32)
    return _clamp_pil(arr)


def shot_noise(img, severity):
    lam = [60, 25, 12, 5, 3][severity - 1]
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = np.random.poisson(arr * lam).astype(np.float32) / lam
    return _clamp_pil(arr)


def impulse_noise(img, severity):
    prob = [0.02, 0.04, 0.08, 0.12, 0.18][severity - 1]
    arr = np.asarray(img).astype(np.float32) / 255.0
    mask = np.random.choice([0, 1, 2], size=arr.shape[:2], p=[1 - prob, prob / 2, prob / 2])
    arr[mask == 1] = 0.0
    arr[mask == 2] = 1.0
    return _clamp_pil(arr)


def speckle_noise(img, severity):
    sigma = [0.10, 0.20, 0.35, 0.50, 0.65][severity - 1]
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr += arr * np.random.normal(0, sigma, arr.shape).astype(np.float32)
    return _clamp_pil(arr)


# Blur corruptions
def gauss_blur(img, severity):
    sigma = [0.5, 1.0, 1.5, 2.5, 3.5][severity - 1]
    arr = np.asarray(img).astype(np.float32) / 255.0
    for c in range(3):
        arr[:, :, c] = gaussian_filter(arr[:, :, c], sigma=sigma)
    return _clamp_pil(arr)


def pixelate(img, severity):
    factor = [0.8, 0.6, 0.4, 0.3, 0.2][severity - 1]
    w, h = img.size
    small = img.resize((max(1, int(w * factor)), max(1, int(h * factor))), Image.BOX)
    return small.resize((w, h), Image.NEAREST)


# Brightness / Contrast
def brightness_up(img, severity):
    factor = [1.2, 1.4, 1.6, 1.9, 2.2][severity - 1]
    return ImageEnhance.Brightness(img).enhance(factor)


def brightness_down(img, severity):
    factor = [0.8, 0.6, 0.4, 0.25, 0.15][severity - 1]
    return ImageEnhance.Brightness(img).enhance(factor)


def contrast_down(img, severity):
    factor = [0.75, 0.55, 0.35, 0.20, 0.10][severity - 1]
    return ImageEnhance.Contrast(img).enhance(factor)


# Digital corruption
def jpeg_compression(img, severity):
    quality = [80, 60, 40, 20, 10][severity - 1]
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

CORRUPTIONS = {
    "gaussian_noise":   gaussian_noise,
    "shot_noise":       shot_noise,
    "impulse_noise":    impulse_noise,
    "speckle_noise":    speckle_noise,
    "gaussian_blur":    gauss_blur,
    "pixelate":         pixelate,
    "brightness_up":    brightness_up,
    "brightness_down":  brightness_down,
    "contrast_down":    contrast_down,
    "jpeg_compression": jpeg_compression,
}

SEVERITIES = [1, 2, 3, 4, 5]

In [ ]:
@torch.no_grad()
def evaluate_accuracy(model, loader, device):
    model.eval()
    all_confs, all_preds, all_labels = [], [], []
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        conf, pred = probs.max(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
        all_confs.append(conf.cpu())
        all_preds.append(pred.cpu())
        all_labels.append(labels.cpu())
    acc = 100.0 * correct / total
    return acc, torch.cat(all_confs), torch.cat(all_preds), torch.cat(all_labels)


def compute_ece(confs, preds, labels, n_bins=15):
    # Expected calibration error
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs, bin_counts = [], [], []
    for lo, hi in zip(bin_boundaries[:-1], bin_boundaries[1:]):
        mask = (confs >= lo) & (confs < hi)
        if mask.sum() == 0:
            bin_accs.append(0.0)
            bin_confs.append(0.0)
            bin_counts.append(0)
            continue
        acc_in_bin = (preds[mask] == labels[mask]).float().mean().item()
        conf_in_bin = confs[mask].mean().item()
        count = mask.sum().item()
        ece += count * abs(acc_in_bin - conf_in_bin)
        bin_accs.append(acc_in_bin)
        bin_confs.append(conf_in_bin)
        bin_counts.append(count)
    ece /= len(confs)
    return ece, bin_accs, bin_confs, bin_counts

In [ ]:
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

clean_results = {}
for name, model in trained_models.items():
    acc, confs, preds, labels = evaluate_accuracy(model, val_loader, DEVICE)
    ece, *_ = compute_ece(confs, preds, labels)
    clean_results[name] = {"acc": acc, "ece": ece, "confs": confs, "preds": preds, "labels": labels}
    print(f"{name}  Clean Acc = {acc:.2f}%   ECE = {ece:.4f}")

In [ ]:
corruption_results = {s: {} for s in STRATEGIES}

total_evals = len(STRATEGIES) * len(CORRUPTIONS) * len(SEVERITIES)

for corr_name, corr_fn in CORRUPTIONS.items():
    for sev in SEVERITIES:
        corrupt_transform = lambda img, _fn=corr_fn, _s=sev: _fn(img, _s)
        corr_ds = CorruptedDataset(val_dataset_raw, corrupt_transform, tensor_transform)
        corr_loader = DataLoader(
            corr_ds, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True,
        )
        for strat_name, model in trained_models.items():
            acc, confs, preds, labels = evaluate_accuracy(model, corr_loader, DEVICE)
            ece, *_ = compute_ece(confs, preds, labels)
            corruption_results[strat_name].setdefault(corr_name, {})[sev] = {
                "acc": acc, "ece": ece,
            }


In [ ]:
save_results = {
    "clean": {s: {"acc": v["acc"], "ece": v["ece"]} for s, v in clean_results.items()},
    "corrupted": corruption_results,
}
with open(os.path.join(RESULTS_DIR, "evaluation_results.json"), "w") as f:
    json.dump(save_results, f, indent=2)

In [ ]:
# Summary df
rows = []
for strat in STRATEGIES:
    for corr_name in CORRUPTIONS:
        for sev in SEVERITIES:
            r = corruption_results[strat][corr_name][sev]
            rows.append({
                "strategy": strat,
                "corruption": corr_name,
                "severity": sev,
                "acc": r["acc"],
                "ece": r["ece"],
            })
df = pd.DataFrame(rows)

# Mean accuracy per corruption type
mean_by_corr = df.groupby(["strategy", "corruption"])["acc"].mean().reset_index()
mean_by_corr_pivot = mean_by_corr.pivot(index="corruption", columns="strategy", values="acc")
mean_by_corr_pivot = mean_by_corr_pivot[STRATEGIES]

# Mean Corruption Error — average accuracy drop vs clean
for strat in STRATEGIES:
    clean_acc = clean_results[strat]["acc"]
    mean_by_corr_pivot[f"{strat}_drop"] = clean_acc - mean_by_corr_pivot[strat]

# Overall summary
print("==================================================================")
print("Mean Accuracy per Corruption (averaged over severities)")
print("==================================================================")
print(mean_by_corr_pivot[STRATEGIES].round(2).to_string())

print("\n")
print("==================================================================")
print("Overall Summary")
print("==================================================================")
for strat in STRATEGIES:
    clean_acc = clean_results[strat]["acc"]
    mean_corr_acc = df[df.strategy == strat]["acc"].mean()
    mce = clean_acc - mean_corr_acc
    clean_ece = clean_results[strat]["ece"]
    mean_corr_ece = df[df.strategy == strat]["ece"].mean()
    print(f"{strat}  Clean: {clean_acc:.2f}%  Mean Corr: {mean_corr_acc:.2f}%  mCE: {mce:.2f}pp  ECE(clean): {clean_ece:.4f}  ECE(corr): {mean_corr_ece:.4f}")

  Mean Accuracy per Corruption (averaged over severities)
strategy          baseline  randaugment  augmix
corruption                                     
brightness_down      41.49        56.45   50.03
brightness_up        46.61        58.15   51.88
contrast_down        32.23        49.74   38.97
gaussian_blur        15.14        15.40   22.01
gaussian_noise       25.01        27.08   32.57
impulse_noise        13.35        29.73   30.22
jpeg_compression     42.30        43.34   47.41
pixelate             19.33        17.34   20.73
shot_noise           15.37        18.37   25.03
speckle_noise        23.89        28.38   34.36


  Overall Summary
  baseline         Clean: 61.66%  Mean Corr: 27.47%  mCE: 34.19pp  ECE(clean): 0.1667  ECE(corr): 0.3934
  randaugment      Clean: 62.14%  Mean Corr: 34.40%  mCE: 27.74pp  ECE(clean): 0.1935  ECE(corr): 0.3354
  augmix           Clean: 61.64%  Mean Corr: 35.32%  mCE: 26.32pp  ECE(clean): 0.1864  ECE(corr): 0.3152


In [ ]:
fig, axes = plt.subplots(1, len(STRATEGIES), figsize=(7 * len(STRATEGIES), 6), sharey=True)

for ax, strat in zip(axes, STRATEGIES):
    matrix = []
    for corr_name in CORRUPTIONS:
        row = [corruption_results[strat][corr_name][s]["acc"] for s in SEVERITIES]
        matrix.append(row)
    matrix = np.array(matrix)
    sns.heatmap(
        matrix, ax=ax, annot=True, fmt=".1f",
        xticklabels=SEVERITIES,
        yticklabels=list(CORRUPTIONS.keys()),
        vmin=0, vmax=100, cbar_kws={"label": "Accuracy (%)"},
    )
    ax.set_title(f"{strat}", fontsize=14)
    ax.set_xlabel("Severity")
    if ax == axes[0]:
        ax.set_ylabel("Corruption")

plt.suptitle("Accuracy under Corruptions")
plt.savefig(os.path.join(RESULTS_DIR, "corruption_heatmap.png"))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(CORRUPTIONS))
width = 0.25
colors = {"baseline": "C0", "randaugment": "C1", "augmix": "C2"}

for i, strat in enumerate(STRATEGIES):
    accs = [mean_by_corr_pivot.loc[c, strat] for c in CORRUPTIONS]
    bars = ax.bar(x + i * width, accs, width, label=strat, color=colors[strat], alpha=0.85)

# Clean accuracy lines
for i, strat in enumerate(STRATEGIES):
    ax.axhline(clean_results[strat]["acc"], color=colors[strat], ls="--", alpha=0.5,
               label=f"{strat} clean" if i == 0 else None)

ax.set_xticks(x + width)
ax.set_xticklabels(list(CORRUPTIONS.keys()))
ax.set_ylabel("Accuracy (%)")
ax.set_title("Mean Corruption Accuracy by Type (avg over severities)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.savefig(os.path.join(RESULTS_DIR, "corruption_bar_chart.png"))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for strat in STRATEGIES:
    mean_acc_by_sev = [
        np.mean([corruption_results[strat][c][s]["acc"] for c in CORRUPTIONS])
        for s in SEVERITIES
    ]
    ax.plot(SEVERITIES, mean_acc_by_sev, "o-", label=strat, color=colors[strat], linewidth=2)
    # Clean accuracy lines
    ax.axhline(clean_results[strat]["acc"], color=colors[strat], ls="--", alpha=0.4)

ax.set_xlabel("Severity Level")
ax.set_ylabel("Mean Accuracy (%)")
ax.set_title("Accuracy Degradation vs Corruption Severity")
ax.set_xticks(SEVERITIES)
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig(os.path.join(RESULTS_DIR, "severity_curve.png"))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for strat in STRATEGIES:
    mean_ece_by_sev = [
        np.mean([corruption_results[strat][c][s]["ece"] for c in CORRUPTIONS])
        for s in SEVERITIES
    ]
    ax.plot(SEVERITIES, mean_ece_by_sev, "o-", label=strat, color=colors[strat], linewidth=2)
    ax.axhline(clean_results[strat]["ece"], color=colors[strat], ls="--", alpha=0.4)

ax.set_xlabel("Severity Level")
ax.set_ylabel("ECE")
ax.set_title("Calibration Error vs Corruption Severity")
ax.set_xticks(SEVERITIES)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "ece_vs_severity.png"))
plt.show()